# 单臂重建（对齐 renderer 官方变换链）

这个 Notebook 将 `Version4` 里前置读取逻辑整合为一个 cell，并提供一个与 `renderer.py` 同构的点云/机器人对齐可视化 cell。

## 1) 整合前置逻辑（导入 + 配置 + 数据读取 + FK）

In [3]:
import os
import h5py
import numpy as np
import open3d as o3d
import k3d
from scipy.spatial.transform import Rotation
from omegaconf import OmegaConf

from airexo.helpers.urdf_robot import forward_kinematic_single
from airexo.helpers.constants import ROBOT_PREDEFINED_TRANSFORMATION, O3D_RENDER_TRANSFORMATION

# ===== 数据路径配置 =====
SCENE_PATH = "/data/haoxiang/data/FLIPPING_v3/train/scene_0001"
LOWDIM_H5_PATH = os.path.join(SCENE_PATH, "lowdim/lowdim.h5")
URDF_FILE = "airexo/urdf_models/robot/left_robot_inhand.urdf"
JOINT_CFG_PATH = "airexo/configs/joint/left/robot.yaml"

# ===== 标定数据（沿用 Version4）=====
CALIB_DATA = {
    "is_global": True,
    "pose_in_link": [
        0.07783932332093665,
        0.2078814260418823,
        0.34723683952957585,
        0.2273133855008057,
        -0.6785647482083789,
        0.6637673415778982,
        -0.21746591345696367
    ],
    "error": 0.0024148397685646483,
    "parent_link_name": "world",
    "cam_serial": "104122060902",
    "intrinsics": [
        [915.384521484375, 0.0, 633.3715209960938],
        [0.0, 914.9421997070312, 354.1505432128906],
        [0.0, 0.0, 1.0]
    ]
}

class SingleRobotData:
    """单臂机器人数据加载器（沿用 Version4）"""

    def __init__(self, lowdim_h5_path, calib_data, joint_cfg_path, urdf_file):
        self.lowdim_h5_path = lowdim_h5_path
        self.calib_data = calib_data
        self.urdf_file = urdf_file
        self.joint_cfgs = OmegaConf.load(joint_cfg_path)
        self._load_lowdim_data()
        self._process_calibration()

    def _load_lowdim_data(self):
        print(f"📂 加载 lowdim 数据: {self.lowdim_h5_path}")
        with h5py.File(self.lowdim_h5_path, 'r') as f:
            self.joint_positions = f['joint_position_rad_062046'][:]
            self.ee_states = f['ee_state_062046'][:]
            self.timestamps = f['timestamp'][:]
            if 'tcp_pose_062046' in f:
                self.tcp_poses = f['tcp_pose_062046'][:]
        print(f"✓ 帧数: {len(self.timestamps)}")

    def _process_calibration(self):
        """pose_in_link: [x,y,z,qx,qy,qz,qw], 表示相机在机器人 base 下位姿"""
        self.intrinsic = np.array(self.calib_data['intrinsics'], dtype=np.float32)
        pose_in_link = self.calib_data['pose_in_link']
        position = np.array(pose_in_link[:3], dtype=np.float32)
        quaternion_wxyz = np.array(pose_in_link[3:], dtype=np.float32)
        quaternion = np.array([quaternion_wxyz[1], quaternion_wxyz[2], quaternion_wxyz[3], quaternion_wxyz[0]], dtype=np.float32)

        R = Rotation.from_quat(quaternion).as_matrix().astype(np.float32)
        self.cam_to_base = np.eye(4, dtype=np.float32)
        self.cam_to_base[:3, :3] = R
        self.cam_to_base[:3, 3] = position
        self.base_to_cam = np.linalg.inv(self.cam_to_base).astype(np.float32)

        print("🎯 标定完成")
        print("cam_to_base:")
        print(self.cam_to_base)

    def get_joint_at_timestamp(self, timestamp_idx):
        joint_angles = self.joint_positions[timestamp_idx]
        ee_state = self.ee_states[timestamp_idx, 0]
        return np.concatenate([joint_angles, [ee_state]], axis=0)


def reconstruct_fk_transforms_at_frame(robot_data, frame_idx):
    """只做 FK，返回 link 变换和 visuals_map；不提前对 mesh 做 cam/base 相关变换。"""
    joint_state = robot_data.get_joint_at_timestamp(frame_idx)
    transforms, visuals_map = forward_kinematic_single(
        joint=joint_state,
        joint_cfgs=robot_data.joint_cfgs,
        is_rad=True,
        urdf_file=robot_data.urdf_file,
        with_visuals_map=True
    )
    return transforms, visuals_map


robot_data = SingleRobotData(
    lowdim_h5_path=LOWDIM_H5_PATH,
    calib_data=CALIB_DATA,
    joint_cfg_path=JOINT_CFG_PATH,
    urdf_file=URDF_FILE
)

print("\n✅ 前置逻辑已就绪（读取/标定/FK函数）")

📂 加载 lowdim 数据: /data/haoxiang/data/FLIPPING_v3/train/scene_0001/lowdim/lowdim.h5
✓ 帧数: 13036
🎯 标定完成
cam_to_base:
[[ 0.02424297 -0.8019524   0.5968958   0.07783932]
 [-0.99968404 -0.01548306  0.01980016  0.20788142]
 [-0.00663701 -0.5971872  -0.8020744   0.34723684]
 [ 0.          0.          0.          1.        ]]

✅ 前置逻辑已就绪（读取/标定/FK函数）


## 2) 修正变换逻辑的 K3D 对齐 cell（对齐 renderer 官方）

In [4]:
def debug_alignment_k3d_renderer_aligned(robot_data, frame_idx, rgb_path, depth_path, subsample_step=4):
    """
    变换链严格对齐 renderer.py / visualizer.py：
    tf = O3D_RENDER_TRANSFORMATION @ cam_to_base @ ROBOT_PREDEFINED_TRANSFORMATION @ link_tf @ visual_offset

    点云也统一放到 Open3D 渲染空间：
    pcd_cam.transform(O3D_RENDER_TRANSFORMATION)
    """
    print("☁️ 读取 RGBD 并生成点云（renderer 对齐）...")

    # 1) 点云：相机系 RGBD -> Open3D 渲染空间
    rgb_img = o3d.io.read_image(rgb_path)
    depth_img = o3d.io.read_image(depth_path)

    rgbd = o3d.geometry.RGBDImage.create_from_color_and_depth(
        rgb_img,
        depth_img,
        depth_scale=1000.0,
        convert_rgb_to_intensity=False
    )

    K = robot_data.intrinsic
    fx, fy = float(K[0, 0]), float(K[1, 1])
    cx, cy = float(K[0, 2]), float(K[1, 2])

    h, w = np.asarray(rgb_img).shape[:2]
    intrinsic_o3d = o3d.camera.PinholeCameraIntrinsic(
        width=int(w), height=int(h), fx=fx, fy=fy, cx=cx, cy=cy
    )

    pcd_cam = o3d.geometry.PointCloud.create_from_rgbd_image(rgbd, intrinsic_o3d)
    pcd_cam.transform(O3D_RENDER_TRANSFORMATION)

    pcd_points = np.asarray(pcd_cam.points).astype(np.float32)
    pcd_colors = np.asarray(pcd_cam.colors).astype(np.float32)

    valid = np.isfinite(pcd_points).all(axis=1)
    pcd_points = pcd_points[valid]
    pcd_colors = pcd_colors[valid]

    # K3D 性能优化
    if subsample_step > 1:
        pcd_points = pcd_points[::subsample_step]
        pcd_colors = pcd_colors[::subsample_step]

    pcd_colors_u32 = (np.clip(pcd_colors, 0.0, 1.0) * 255).astype(np.uint32)
    pcd_colors_packed = (pcd_colors_u32[:, 0] << 16) | (pcd_colors_u32[:, 1] << 8) | pcd_colors_u32[:, 2]

    # 2) FK
    print("🤖 计算 FK 并加载 mesh（renderer 对齐变换链）...")
    transforms, visuals_map = reconstruct_fk_transforms_at_frame(robot_data, frame_idx)

    # 3) K3D 绘制
    print("🎨 构建 K3D 场景...")
    plot = k3d.plot(background_color=0xFFFFFF)

    # 点云
    plot += k3d.points(
        positions=pcd_points,
        colors=pcd_colors_packed.astype(np.uint32),
        point_size=0.002,
        shader='flat',
        name='Point Cloud (O3D render space)'
    )

    # 机器人 mesh
    urdf_dir = os.path.dirname(robot_data.urdf_file)
    mesh_count = 0

    for link, transform in transforms.items():
        if link not in visuals_map:
            continue

        for visual in visuals_map[link]:
            if visual.geom_param is None:
                continue

            mesh_rel = visual.geom_param[0] if isinstance(visual.geom_param, (list, tuple)) else visual.geom_param
            mesh_path = os.path.join(urdf_dir, mesh_rel)
            if not os.path.exists(mesh_path):
                continue

            mesh_o3d = o3d.io.read_triangle_mesh(mesh_path)

            # ✅ 官方变换链（与 renderer.py 一致）
            tf = (
                O3D_RENDER_TRANSFORMATION
                @ robot_data.cam_to_base
                @ ROBOT_PREDEFINED_TRANSFORMATION
                @ transform.matrix()
                @ visual.offset.matrix()
            )
            mesh_o3d.transform(tf)

            verts = np.asarray(mesh_o3d.vertices).astype(np.float32)
            faces = np.asarray(mesh_o3d.triangles).astype(np.uint32)
            if len(verts) == 0 or len(faces) == 0:
                continue

            plot += k3d.mesh(
                vertices=verts,
                indices=faces,
                color=0xFF4444,
                opacity=0.65,
                name=f"{link}:{os.path.basename(str(mesh_rel))}"
            )
            mesh_count += 1

    # 坐标轴（Open3D 渲染空间原点）
    axis_size = 0.2
    plot += k3d.vectors(
        origins=[[0, 0, 0], [0, 0, 0], [0, 0, 0]],
        vectors=[[axis_size, 0, 0], [0, axis_size, 0], [0, 0, axis_size]],
        colors=[0xff0000, 0x00ff00, 0x0000ff],
        line_width=0.01
    )

    plot.display()
    print(f"✅ 完成：点云 {len(pcd_points)} 点，机器人 mesh {mesh_count} 个，均在 O3D 渲染空间。")
    return plot


# ===== 运行调试 =====
rgb_path = "/data/haoxiang/data/FLIPPING_v3/train/scene_0001/cam_104122060902/color/1767593840262.png"
depth_path = "/data/haoxiang/data/FLIPPING_v3/train/scene_0001/cam_104122060902/depth/1767593840262.png"

FRAME_IDX = 0
plot = debug_alignment_k3d_renderer_aligned(
    robot_data=robot_data,
    frame_idx=FRAME_IDX,
    rgb_path=rgb_path,
    depth_path=depth_path,
    subsample_step=4
)

☁️ 读取 RGBD 并生成点云（renderer 对齐）...
🤖 计算 FK 并加载 mesh（renderer 对齐变换链）...
🎨 构建 K3D 场景...


Output()

✅ 完成：点云 205961 点，机器人 mesh 17 个，均在 O3D 渲染空间。
